# Phase 7 — End-to-End Pipeline

Wires every module together into a single scoring run.

```
Scoring population (8M ETB base)
         │
         ▼
 [1] Behavioral Segmentation
         │
         ▼
 [2] Feature Selection  (apply pre-fitted selector)
         │
         ▼
 [3] Income Estimation
     ├── PAYROLL  → verified income (bypass)
     └── Others   → SegmentEnsemble (best model per segment)
         │
         ▼
 [4] Business Confidence Index (BCI)
         │
         ▼
 [5] Affordability Engine (BOT DSCR norms)
         │
         ▼
 [6] Policy Engine
     STP_APPROVE | STP_DECLINE | REFER_INCOME_VERIFY | MANUAL_REVIEW
```

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.segmentation import SegmentationPipeline
from src.income_estimation import IncomeEstimationPipeline
from src.feature_selection import FeatureSelectionPipeline
from src.modeling import LabelEngineer, SegmentModelTrainer
from src.modeling.ensemble import SegmentEnsemble
from src.bci import BCIScorer
from src.affordability import AffordabilityEngine, PolicyEngine
from src.utils import setup_logging
from data.sample.generate_sample import SampleDataGenerator

setup_logging('INFO')
SEED = 42

## Generate Sample Data

In [ ]:
gen = SampleDataGenerator(seed=SEED)

# Training data (160K verified records)
X_train, y_train = gen.training_data(n_customers=5000)  # Use 5K for demo speed

# Scoring population (ETB base)
X_score = gen.scoring_population(n_customers=10000)

print(f'Training: {X_train.shape}  |  Scoring: {X_score.shape}')
print(f'Segment mix (scoring):\n{X_score.segment.value_counts()}')

## Step 1 — Segmentation

In [ ]:
seg_pipeline = SegmentationPipeline(config_path='../config/config.yaml')
X_score_seg = seg_pipeline.run(X_score)
print(seg_pipeline.get_summary(X_score_seg))

## Step 2 — Feature Selection (fit on train, apply to score)

In [ ]:
fs = FeatureSelectionPipeline(
    run_boruta=False,     # Fast mode
    run_stability=False,
)
fs.fit(X_train, y_train, segment_col='segment')
print(f'Selected {len(fs.final_features_)} features')

# Apply to both train and score
feat_cols = fs.final_features_

## Step 3 — Advanced Income Estimation

In [ ]:
# Label engineering
label_eng = LabelEngineer()
label_eng.fit(X_train, y_train, segment_col='segment')

# Segment model trainer
trainer = SegmentModelTrainer(
    label_engineer=label_eng,
    feature_selector=fs,
    cv_folds=3,
    lgb_losses=['huber_10k', 'quantile_p40', 'log_rmse'],
    label_strategies=['raw', 'robust', 'shrunk_composite'],
    include_lstm=False,
    include_tabpfn=False,
    max_rows_per_segment=2000,
)
trainer.fit(X_train, y_train, segment_col='segment')

print('Best per segment:')
for seg, best in trainer.best_per_segment_.items():
    print(f"  {seg}: {best['model']} | {best['loss']} | MAE={best['cv_mae']:,.0f} THB")

In [ ]:
# Score the population — handle PAYROLL bypass
income_results = pd.DataFrame(index=X_score_seg.index)

payroll_mask = X_score_seg['segment'] == 'PAYROLL'
income_results.loc[payroll_mask, 'income_source'] = 'PAYROLL'
income_results.loc[payroll_mask, 'income_estimate'] = X_score_seg.loc[payroll_mask, 'payroll_income']
income_results.loc[payroll_mask, 'model_confidence'] = 1.0
income_results.loc[payroll_mask, 'income_interval_width'] = 0.0

non_payroll_mask = ~payroll_mask
income_preds = trainer.predict(X_score_seg[non_payroll_mask], segment_col='segment')
income_results.loc[non_payroll_mask, 'income_source'] = 'ESTIMATED'
income_results.loc[non_payroll_mask, 'income_estimate'] = income_preds.values
income_results.loc[non_payroll_mask, 'model_confidence'] = 0.70   # Placeholder
income_results.loc[non_payroll_mask, 'income_interval_width'] = income_preds.values * 0.30

income_results['income_band'] = pd.cut(
    income_results['income_estimate'],
    bins=[0, 15000, 30000, 50000, 100000, 1e9],
    labels=['<15K', '15-30K', '30-50K', '50-100K', '>100K']
)

print(f'Income estimates: median={income_results.income_estimate.median():,.0f} THB')
print(income_results.income_band.value_counts())

## Step 4 — BCI

In [ ]:
bci_scorer = BCIScorer(config_path='../config/config.yaml')
bci_results = bci_scorer.compute(X_score_seg, income_results, segment_col='segment')
print(bci_scorer.get_summary(bci_results))

## Step 5 — Affordability Engine

In [ ]:
aff_engine = AffordabilityEngine(config_path='../config/config.yaml')
aff_results = aff_engine.compute(bci_results, X_score_seg)
print(aff_engine.get_summary(aff_results))

# DSCR stress test
stress = aff_engine.stress_test(aff_results)
print(stress)

## Step 6 — Policy Engine + Final Decisions

In [ ]:
policy = PolicyEngine(config_path='../config/config.yaml')
final_output = policy.get_full_output(
    bci_results, aff_results, income_results, X_score_seg
)

decision_summary = policy.get_decision_summary(policy.decide(bci_results, aff_results))
print('\nFinal Decision Summary:')
print(decision_summary.to_string(index=False))

## Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Segment distribution
X_score_seg['segment'].value_counts().plot(kind='bar', ax=axes[0,0], title='Segment Distribution', color='steelblue')
axes[0,0].tick_params(axis='x', rotation=30)

# 2. Income distribution by segment
plot_df = final_output[final_output.income_estimate_raw < 200000]
plot_df.groupby('segment')['income_estimate_raw'].median().sort_values().plot(
    kind='barh', ax=axes[0,1], title='Median Income Estimate by Segment', color='teal')

# 3. BCI distribution
bci_results['bci_score'].hist(bins=30, ax=axes[0,2], color='orange', edgecolor='white')
axes[0,2].set_title('BCI Score Distribution')
axes[0,2].axvline(60, color='red', linestyle='--', label='MEDIUM')
axes[0,2].axvline(80, color='green', linestyle='--', label='HIGH')
axes[0,2].legend()

# 4. Final decisions
final_output['final_decision'].value_counts().plot(
    kind='pie', ax=axes[1,0], title='Policy Decisions', autopct='%1.0f%%')

# 5. ADSC distribution
final_output[final_output.adsc.between(-50000, 100000)]['adsc'].hist(
    bins=40, ax=axes[1,1], color='purple', edgecolor='white')
axes[1,1].axvline(0, color='red', linestyle='--')
axes[1,1].set_title('Available Debt Service Capacity (ADSC) Distribution')

# 6. BCI × Affordability matrix
cross = pd.crosstab(final_output['bci_band'], final_output['final_decision'])
import seaborn as sns
sns.heatmap(cross, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1,2])
axes[1,2].set_title('BCI Band × Policy Decision')

plt.suptitle('End-to-End Income & Affordability Pipeline — Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nTotal customers scored: {len(final_output):,}')
print(f'STP Approve: {(final_output.final_decision == "STP_APPROVE").mean():.1%}')
print(f'STP Decline: {(final_output.final_decision == "STP_DECLINE").mean():.1%}')
print(f'Refer:       {(final_output.final_decision.str.startswith("REFER")).mean():.1%}')